Put logo

## Curve Bootstrapping

In [ ]:
%load_ext autoreload
%autoreload 2

#imports
import datetime as dt
from dateutil.relativedelta import relativedelta
import pandas as pd
import matplotlib.pyplot as plt
from rivapy.marketdata.bootstrapping_2025 import bootstrap_curve, get_quote
from rivapy.instruments.deposit_specifications import DepositSpecification
from rivapy.instruments.fra_specifications import ForwardRateAgreementSpecification
from rivapy.instruments.ir_swap_specification import InterestRateSwapSpecification, IrFixedLegSpecification, IrFloatLegSpecification
from rivapy.instruments.notional_structure import ConstNotionalStructure
from rivapy.tools.enums import DayCounterType, InterpolationType, ExtrapolationType

import matplotlib.pyplot as plt
import math
from rivapy.tools.datetools import DayCounter
from rivapy.tools._validators import print_member_values
from rivapy.pricing.deposit_pricing import DepositPricer
from rivapy.pricing.interest_rate_swap_pricing import InterestRateSwapPricer

In [ ]:
#set up instruments
#set up deposit
#set up ir_swap



### Setting up deposits / fixings

The fixing of the underlying reference rate is published daily and represents a certain average rate earned over a period corresponding to the tenor. The rate is calculated from quotes obtained from a panel of selected banks. The start date (spot date) of the period can deviate from the fixing date. This difference is referred to as the spot lag. 

The fixing is usually used as the starting point for the bootstrapping of forward curves. It can be specified as a deposit in pyvacon.

In [ ]:
##########################################
# setting up depoist
# calculation date
ref_date = dt.datetime(2019, 8, 31)

# start date of the accrual period with spot lag equal to 2 days
start_date = ref_date + dt.timedelta(days=2)

# end date of the accrual period is 1 day after startdate
end_date = start_date + dt.timedelta(days=1)

# specification of the deposit
# deposit = pyvacon.finance.specification.DepositSpecification(
#     'OVERNIGHT_DEPOSIT', 'dummy_issuer', 'NONE', 'EUR',
#     refdate, startdate, enddate, 100, 0.01, 'Act365Fixed')
ccy = "EUR"
dcc = "Act365Fixed"#"Act360"#"Act365Fixed"#"Act360"
rate = 0.01
notional = 100.0
deposit = DepositSpecification(
    obj_id="OVERNIGHT_DEPOSIT",
    issuer="dummy_issuer",
    currency=ccy,
    fixing_date=ref_date,
    start_date=start_date,
    maturity_date=end_date,
    notional=notional,
    rate=rate,
    day_count_convention=dcc,
)

# check dates, deposit start date cannot be the same as  maturity date
print(f"ref date: {ref_date}")
print(f"start date: {start_date} with spot lag 2 days")
print(f"end date: {end_date}")
print(f"adjusted start date: {deposit.start_date}")
print(f"adjusted end date: {deposit.maturity_date}")

print_member_values(deposit)



end_date_deposits = [
    start_date + dt.timedelta(days=1),
    start_date + dt.timedelta(days=7),
    start_date + dt.timedelta(days=30),
    start_date + dt.timedelta(days=60),
    start_date + dt.timedelta(days=90),
    start_date + dt.timedelta(days=180),
    start_date + dt.timedelta(days=270),
    start_date + dt.timedelta(days=360),
]

quotes_deposits = [0.025,0.028, 0.0283, 0.029, 0.0305, 0.0315, 0.0348, 0.05]
epsilon = 0#0.007
other_deposits = []

for i in range(len(quotes_deposits)):
    temp_deposit = DepositSpecification(
        obj_id="DEPOSIT_" + str(i + 1),
        issuer="dummy_issuer",
        currency=ccy,
        fixing_date=ref_date,
        start_date=start_date,
        maturity_date=end_date_deposits[i],
        notional=notional,
        rate=quotes_deposits[i] + epsilon,
        day_count_convention=dcc,
    )
    other_deposits.append(temp_deposit)

In [ ]:
print(f"ref date: {ref_date}")
print(f"start date: {start_date} with spot lag 2 days")
print(f"end date: {end_date}")
print(f"adjusted start date: {deposit.start_date}")
print(f"adjusted end date: {deposit.maturity_date}")

In [ ]:
#additional set of test deposits taken from inputQuotes.csv
#RiVaPy\notebooks\marketdata\inputQuotes.csv
end_date_deposits_csv = [
    start_date + dt.timedelta(days=1),
    start_date + dt.timedelta(days=90),
    start_date + dt.timedelta(days=180),
    start_date + dt.timedelta(days=270),
]

#quotes_deposits_csv = [-0.00345, -0.00329, -0.00327672, -0.00323786] #why in teh data it is negative?
quotes_deposits_csv = [0.00323786, 0.00327672, 0.00329, 0.00345] #sorted to be increasing with maturity...
other_deposits_csv = []

for i in range(len(quotes_deposits_csv)):
    temp_deposit = DepositSpecification(
        obj_id="DEPOSIT_" + str(i + 1),
        issuer="dummy_issuer",
        currency=ccy,
        fixing_date=ref_date,
        start_date=start_date,
        maturity_date=end_date_deposits[i],
        notional=notional,
        rate=0.01,#the rate here should not matter as impliedSImplyCompoundedrate does not care
        day_count_convention=dcc,
    )
    other_deposits_csv.append(temp_deposit)

In [ ]:
# #TOGGLE THIS FOR TESTING
# quotes_deposits = quotes_deposits_csv 
# other_deposits = other_deposits_csv
# end_date_deposits = end_date_deposits_csv


instruments = other_deposits
quotes = quotes_deposits

# instruments = other_deposits_csv
# quotes = quotes_deposits_csv
# estr = bootstrap_curve(ref_date, 'ESTR_DC', DayCounterType.ACT360,  
#                        instruments, quotes, interpolation_type=InterpolationType.LINEAR_LOG)

estr = bootstrap_curve(ref_date, 'ESTR_DC', DayCounterType.Act365Fixed,  
                      instruments, quotes, interpolation_type=InterpolationType.LINEAR_LOG)

In [ ]:
# # sample 3M EURIBOR curve with ois discounting (the same instruments are used for simplification)
# quotes = [0.003, 0.0075]

# euribor_3m = bootstrap_curve(ref_date, 'EUR3M_DC', 'Act365Fixed', instruments, quotes, estr)

In [ ]:
# # sample 6M EURIBOR curve with ois bootstrapping and the 3M EURIBOR curve as the basis index
# quotes = [0.003, 0.006]
# # the basis swap is used instead of the ir swap
# instruments[1] = basis_swap #TODO basis swaps are not yet implemented
# euribor_6m = bootstr.bootstrap_curve(refdate, 'EUR6M_DC', 'Act365Fixed', instruments, quotes, estr, euribor_3m)

In [ ]:
# estr.plot()
# euribor_3m.plot()
# #euribor_6m.plot()

In [ ]:
# plt.figure(1)
# estr.plot(discount_factors=True)
# plt.xlabel("year")
# plt.ylabel("DF")
# plt.legend()

# plt.figure(2)
# estr.plot(discount_factors=False)
# plt.xlabel("year")
# plt.ylabel("zero rate")
# plt.legend()

plt.figure(3)
dates_final = estr.get_dates()
df_final = estr.get_df()

# calculated df from dates
dates_new = [dates_final[0]]
days = 10  # create a smoother curve by adding dates in between
for i in range(1, len(dates_final)):
    while dates_new[-1] + dt.timedelta(days=days) < dates_final[i]:
        dates_new.append(dates_new[-1] + dt.timedelta(days=days))
    dates_new.append(dates_final[i])

#print(dates_new)
values = [estr.rivapy_value(estr.refdate, d) for d in dates_new]
#print(values)

plt.plot(dates_final, df_final, marker="^", label="bootstrapped")

plt.plot(dates_new, values, label="interpolated")

plt.xlabel("year")
plt.ylabel("DF")
plt.legend()

# continuous compounding
plt.figure(4)
dcc = DayCounter(estr.daycounter)
zero_final = []
for i in range(1, len(df_final)):
    delta_t = dcc.yf(estr.refdate, dates_final[i])  # float((dates_new[i] - estr.refdate).days) / 365.0
    zero_final.append(-math.log(df_final[i]) / delta_t)
zero_final.insert(0, zero_final[0])  # adding first entry to account for division by zero for dt

zero_interp = []
for i in range(1, len(values)):
    delta_t = dcc.yf(estr.refdate, dates_new[i])  # float((dates_new[i] - estr.refdate).days) / 365.0
    zero_interp.append(-math.log(values[i]) / delta_t)
zero_interp.insert(0, zero_interp[0])

plt.plot(dates_final, zero_final, marker="^", label="bootstrapped")
#plt.scatter(end_date_deposits, quotes_deposits)
plt.plot(dates_new, zero_interp, label="interpolated")

plt.xlabel("year")
plt.ylabel("zero rates")
plt.legend()
plt.show()


In [ ]:
len(dates_new)
for i in dates_new:
    print(i)

In [ ]:
dc_dates= estr.get_dates()
print(dc_dates)
print(len(dc_dates))
for i in dc_dates:
    print(i)

In [ ]:
end_date_deposits
for i in end_date_deposits:
    print(i)

In [ ]:
print(len(end_date_deposits))
print(len(dates_final))

In [ ]:
for i in range(len(quotes_deposits)):
    #print(f"i:{i} date:{dates_final[i+1]}  zero:{zero_final[i+1]} quote:{quotes_deposits[i]}")
    print(f"i:{i} date:{dates_final[i+1]}  zero:{zero_final[i+1]} date:{end_date_deposits[i]} quote:{quotes_deposits[i]}")

#tolerance 1e-6
# i:0 date:2019-09-03 00:00:00  zero:0.05169734842475251 quote:0.051
# i:1 date:2019-09-09 00:00:00  zero:0.051842710069517586 quote:0.0512
# i:2 date:2019-10-02 00:00:00  zero:0.05207793548329917 quote:0.0515
# i:3 date:2019-11-01 00:00:00  zero:0.05227466599231794 quote:0.0518
# i:4 date:2019-12-02 00:00:00  zero:0.05236399478862491 quote:0.052
# i:5 date:2020-02-28 00:00:00  zero:0.05253685303257888 quote:0.0525
# i:6 date:2020-05-29 00:00:00  zero:0.05249461785244582 quote:0.0528
# i:7 date:2020-08-27 00:00:00  zero:0.05235681622842124 quote:0.053

In [ ]:
print_member_values(other_deposits[5])

In [ ]:
new_df = []
for i in range(len(end_date_deposits)) :
    val = estr.rivapy_value(estr.refdate, end_date_deposits[i])
    new_df.append(val)
    print(val)
    delta_t = dcc.yf(estr.refdate, end_date_deposits[i])
    discount_factor = math.exp(-1*val*delta_t)

In [ ]:
simply_rate = []
for i in range(len(other_deposits)):
    val =DepositPricer.implied_simply_compounded_rate(ref_date, other_deposits[i], estr)
    simply_rate.append(val)

In [ ]:
for i in range(len(quotes_deposits)):
    #per_diff = (zero_final[i+1] - quotes_deposits[i])/quotes_deposits[i] * 100
    #print(f"i:{i} date:{dates_final[i+1]}  percent diff: {per_diff}")

    per_diff = (simply_rate[i] - quotes_deposits[i])/quotes_deposits[i] * 100
    print(f"i:{i} date:{dates_final[i+1]}  percent diff: {per_diff}")


#tolerance 1e-6
# i:0 date:2019-09-03 00:00:00  percent diff: 1.3673498524559047
# i:1 date:2019-09-09 00:00:00  percent diff: 1.255293104526531
# i:2 date:2019-10-02 00:00:00  percent diff: 1.1222048219401421
# i:3 date:2019-11-01 00:00:00  percent diff: 0.916343614513402
# i:4 date:2019-12-02 00:00:00  percent diff: 0.6999899781248362
# i:5 date:2020-02-28 00:00:00  percent diff: 0.07019625253120086
# i:6 date:2020-05-29 00:00:00  percent diff: -0.5783752794586687
# i:7 date:2020-08-27 00:00:00  percent diff: -1.213554285997656

In [ ]:
dc_df= estr.get_df()
for i in range(len(dc_df)):
    print(f"i:{i} date:{dates_final[i]}  DF: {dc_df[i]}")

In [ ]:
continous_compound_DF = []

for i in range(len(end_date_deposits)):
    delta_t = dcc.yf(estr.refdate, end_date_deposits[i])
    discount_factor = math.exp(-1*quotes_deposits[i]*delta_t)
    continous_compound_DF.append(discount_factor)
    
for i in range(len(end_date_deposits)):
    print(f"i:{i} date:{end_date_deposits[i]}  DF: {continous_compound_DF[i]}")

    



In [ ]:
len(end_date_deposits)
for i in end_date_deposits:
    print(i)

In [ ]:
len(continous_compound_DF)

In [ ]:
len(quotes_deposits)

In [ ]:
for i in range(len(continous_compound_DF)):
    per_diff = (dc_df[i+1] - continous_compound_DF[i])/continous_compound_DF[i] * 100
    print(f"i:{i} date:{dates_final[i+1]}  percent diff: {per_diff}")

In [ ]:
annual_compound_DF = []
for i in range(len(end_date_deposits)):
    delta_t = dcc.yf(estr.refdate, end_date_deposits[i])
    discount_factor = 1 / ((1 + quotes_deposits[i]) ** delta_t)
    annual_compound_DF.append(discount_factor)
    
for i in range(len(end_date_deposits)):
    print(f"i:{i} date:{end_date_deposits[i]}  DF: {annual_compound_DF[i]}")


In [ ]:
for i in range(len(annual_compound_DF)):
    per_diff = (dc_df[i+1] - annual_compound_DF[i])/annual_compound_DF[i] * 100
    print(f"i:{i} date:{dates_final[i+1]}  percent diff: {per_diff}")

## SWAPS

### Setting up an interest rate swap
A plain vanilla interest rate swap is a financial contract in which a stream of fixed payments is exchanged for floating payments linked to a reference index. The par rate (r) of a swap is the fixed rate under which the value of the two streams (legs) is equal:
$$ r \cdot \sum_{i=1}^n dcf_{i} \cdot P(0,t_{i} ) = \sum_{k=1}^m F_{k} \cdot dcf_{k} \cdot P(0,t_{k}) $$ 

where $t_{i}$, $i=1,..,n$ and $t_{k}$, $i=1,..,m$ are the payment structures of the fixed and floating legs, and $P(0,t_{i/k})$ are the corresponding discount factors, $dcf_{i/k}$ is the day count fraction for the period $[t_{(i/k-1)},t_{i/k}]$, and $F_{k}$ is the expected value of  underlying reference rate for the period $[t_{(k-1)},t_{k}]$.

The standard payment frequency of the fixed leg depends on the currency of the swap as well as the tenor of the underlying. In the EUR market swaps are usually quoted with annual fixed payments.

The payment frequency of the floating leg usually coincides with the tenor of the underlying reference index. In some currencies, however, the floating rate can be compounded and payed out at less frequent intervals (e.g. CAD). 

In the context of pyvacon an IRS can be defined using an InterestRateSwapSpecification.

In [ ]:
#ref_date = is the same as was in deposits

#1Y maturity swap quarterly payments
# start dates of the accrual periods corresponding to the tenor of the underlying index (3 months). The spot lag is set to 0.
start_dates = [ref_date + relativedelta(months=3*i) for i in range(4)]

# reset dates are equal to start dates if spot lag is 0.
reset_dates = start_dates

# the end dates of the accral periods
end_dates = [x + relativedelta(months=3) for x in start_dates]

# the actual payment dates of the cashflows may differ from the end of the accrual period (e.g. OIS). 
# in the standard case these two sets of dates coincide
pay_dates = end_dates
#notionals = [1.0 for i in range(len(start_dates))] #i.e constant ...

print(end_dates)
ns = ConstNotionalStructure(100.0)
spread = 0.00

# # definition of the floating leg
float_leg =IrFloatLegSpecification(obj_id = 'dummy_float_leg', notional = ns, reset_dates=reset_dates, start_dates=start_dates, end_dates=end_dates,
                                   rate_start_dates=start_dates, rate_end_dates=end_dates, pay_dates=pay_dates, currency = "EUR", 
                                   udl_id="test_udl_id", fixing_id="test_fixing_id", day_count_convention="Act365Fixed", spread=spread)

# # definition of the fixed leg
fixed_leg = IrFixedLegSpecification(fixed_rate = 0.01, obj_id = 'dummy_fixed_leg', notional = 100.0, start_dates=start_dates, 
                                    end_dates=end_dates, pay_dates=pay_dates, currency='EUR', day_count_convention='Act365Fixed')

# # definition of the IR swap
ir_swap = InterestRateSwapSpecification(obj_id="3M_SWAP", notional=ns, issue_date=ref_date, maturity_date=pay_dates[-1],
                                        pay_leg=fixed_leg, receive_leg=float_leg,currency='EUR', day_count_convention="Act365Fixed",
                                        issuer="dummy_issuer", securitization_level="COLLATERALIZED")
 


In [ ]:
#2Y maturity swap quarterly payments
# start dates of the accrual periods corresponding to the tenor of the underlying index (3 months). The spot lag is set to 0.
start_dates2 = [ref_date + relativedelta(months=3*i) for i in range(4*2)]

# reset dates are equal to start dates if spot lag is 0.
reset_dates2 = start_dates2

# the end dates of the accral periods
end_dates2 = [x + relativedelta(months=3) for x in start_dates2]

# the actual payment dates of the cashflows may differ from the end of the accrual period (e.g. OIS). 
# in the standard case these two sets of dates coincide
pay_dates2 = end_dates2
#notionals = [1.0 for i in range(len(start_dates))] #i.e constant ...

ns = ConstNotionalStructure(100.0)
spread = 0.00

# # definition of the floating leg
float_leg2 =IrFloatLegSpecification(obj_id = 'dummy_float_leg2', notional = ns, reset_dates=reset_dates2, start_dates=start_dates2, end_dates=end_dates2,
                                   rate_start_dates=start_dates2, rate_end_dates=end_dates2, pay_dates=pay_dates2, currency = "EUR", 
                                   udl_id="test_udl_id", fixing_id="test_fixing_id", day_count_convention="Act365Fixed", spread=spread)

# # definition of the fixed leg
fixed_leg2 = IrFixedLegSpecification(fixed_rate = 0.01, obj_id = 'dummy_fixed_leg2', notional = 100.0, start_dates=start_dates2, 
                                    end_dates=end_dates2, pay_dates=pay_dates2, currency='EUR', day_count_convention='Act365Fixed')

# # definition of the IR swap
ir_swap2 = InterestRateSwapSpecification(obj_id="3M_SWAP2", notional=ns, issue_date=ref_date, maturity_date=pay_dates2[-1],
                                        pay_leg=fixed_leg2, receive_leg=float_leg2,currency='EUR', day_count_convention="Act365Fixed",
                                        issuer="dummy_issuer", securitization_level="COLLATERALIZED")
 


In [ ]:
#3Y maturity swap quarterly payments
# start dates of the accrual periods corresponding to the tenor of the underlying index (3 months). The spot lag is set to 0.
start_dates3 = [ref_date + relativedelta(months=3*i) for i in range(4*3)]

# reset dates are equal to start dates if spot lag is 0.
reset_dates3 = start_dates3

# the end dates of the accral periods
end_dates3 = [x + relativedelta(months=3) for x in start_dates3]

# the actual payment dates of the cashflows may differ from the end of the accrual period (e.g. OIS). 
# in the standard case these two sets of dates coincide
pay_dates3 = end_dates3
#notionals = [1.0 for i in range(len(start_dates))] #i.e constant ...

ns = ConstNotionalStructure(100.0)
spread = 0.00

# # definition of the floating leg
float_leg3 =IrFloatLegSpecification(obj_id = 'dummy_float_leg3', notional = ns, reset_dates=reset_dates3, start_dates=start_dates3, end_dates=end_dates3,
                                   rate_start_dates=start_dates3, rate_end_dates=end_dates3, pay_dates=pay_dates3, currency = "EUR", 
                                   udl_id="test_udl_id", fixing_id="test_fixing_id", day_count_convention="Act365Fixed", spread=spread)

# # definition of the fixed leg
fixed_leg3 = IrFixedLegSpecification(fixed_rate = 0.01, obj_id = 'dummy_fixed_leg3', notional = 100.0, start_dates=start_dates3, 
                                    end_dates=end_dates3, pay_dates=pay_dates3, currency='EUR', day_count_convention='Act365Fixed')

# # definition of the IR swap
ir_swap3 = InterestRateSwapSpecification(obj_id="3M_SWAP3", notional=ns, issue_date=ref_date, maturity_date=pay_dates3[-1],
                                        pay_leg=fixed_leg3, receive_leg=float_leg3,currency='EUR', day_count_convention="Act365Fixed",
                                        issuer="dummy_issuer", securitization_level="COLLATERALIZED")
 


In [ ]:
# other_swaps = [ir_swap]
# quotes_swaps = [0.01]
# other_swaps = [ir_swap2]
# quotes_swaps = [0.02]
# other_swaps = [ir_swap, ir_swap2]
# quotes_swaps = [0.01, 0.02]
other_swaps = [ir_swap, ir_swap2, ir_swap3]
quotes_swaps = [0.05, 0.06, 0.07]


estr = bootstrap_curve(ref_date, 'ESTR_DC', DayCounterType.Act365Fixed,  
                       other_swaps, quotes_swaps)



In [ ]:

plt.figure(1)
estr.plot(discount_factors=True)
plt.xlabel("year")
plt.ylabel("DF")
plt.legend()

plt.figure(2)
estr.plot(discount_factors=False)
plt.xlabel("year")
plt.ylabel("zero rate")
plt.legend()

In [ ]:
plt.figure(3)
dates_final = estr.get_dates()
df_final = estr.get_df()

# calculated df from dates
dates_new = [dates_final[0]]
days = 10  # create a smoother curve by adding dates in between
for i in range(1, len(dates_final)):
    while dates_new[-1] + dt.timedelta(days=days) < dates_final[i]:
        dates_new.append(dates_new[-1] + dt.timedelta(days=days))
    dates_new.append(dates_final[i])

print(dates_new)
values = [estr.rivapy_value(estr.refdate, d) for d in dates_new]
print(values)

plt.plot(dates_final, df_final, marker="^", label="bootstrapped")
plt.plot(dates_new, values, label="interpolated")

plt.xlabel("year")
plt.ylabel("DF")
plt.legend()

# continuous compounding
plt.figure(4)
dcc = DayCounter(estr.daycounter)
zero_final = []
for i in range(1, len(df_final)):
    delta_t = dcc.yf(estr.refdate, dates_final[i])  # float((dates_new[i] - estr.refdate).days) / 365.0
    zero_final.append(-math.log(df_final[i]) / delta_t)
zero_final.insert(0, zero_final[0])  # adding first entry to account for division by zero for dt

zero_interp = []
for i in range(1, len(values)):
    delta_t = dcc.yf(estr.refdate, dates_new[i])  # float((dates_new[i] - estr.refdate).days) / 365.0
    zero_interp.append(-math.log(values[i]) / delta_t)
zero_interp.insert(0, zero_interp[0])

plt.plot(dates_final, zero_final, marker="^", label="bootstrapped")
# plt.scatter(end_date_deposits, quotes_deposits)
plt.plot(dates_new, zero_interp, label="interpolated")

plt.xlabel("year")
plt.ylabel("zero rates")
plt.legend()
plt.show()

In [ ]:
swap_fair_rate = []
pricing_params = {"fixing_grace_period": 0.0, "set_rate": True, "desired_rate": 1.0}
for i in range(len(other_swaps)):
    instrument_spec = other_swaps[i]
    float_leg = instrument_spec.get_float_leg()
    fixed_leg = instrument_spec.get_fixed_leg()
    yc_discount = estr  # TODO decide how to pass which curves
    yc_forward = estr
    val = InterestRateSwapPricer.compute_swap_rate(ref_date, yc_discount, yc_forward, 
                                                    float_leg, fixed_leg, None, pricing_params)
    print(val)
    swap_fair_rate.append(val)


In [ ]:
for i in range(len(quotes_swaps)):

    per_diff = (swap_fair_rate[i] - quotes_swaps[i])/quotes_swaps[i] * 100
    print(f"i:{i} date:{dates_final[i+1]}  percent diff: {per_diff}")

## DEPOSITS AND SWAPS

In [ ]:
other_swaps = [ir_swap, ir_swap2, ir_swap3]
quotes_swaps = [0.05, 0.06, 0.07]

#we omit the 1 year deposit so as to not overlap with the swap
instruments_both = other_deposits[:-1] + other_swaps
quotes_both = quotes_deposits[:-1] + quotes_swaps
estr_both = bootstrap_curve(ref_date, 'ESTR_DC', DayCounterType.Act365Fixed,  
                       instruments_both, quotes_both)

In [ ]:
plt.figure(1)
estr_both.plot(discount_factors=True)
plt.xlabel("year")
plt.ylabel("DF")
plt.legend()

plt.figure(2)
estr_both.plot(discount_factors=False)
plt.xlabel("year")
plt.ylabel("zero rate")
plt.legend()

[1]


In [ ]:
plt.figure(3)
dates_final = estr_both.get_dates()
df_final = estr_both.get_df()

# calculated df from dates
dates_new = [dates_final[0]]
days = 10  # create a smoother curve by adding dates in between
for i in range(1, len(dates_final)):
    while dates_new[-1] + dt.timedelta(days=days) < dates_final[i]:
        dates_new.append(dates_new[-1] + dt.timedelta(days=days))
    dates_new.append(dates_final[i])

print(dates_new)
values = [estr_both.rivapy_value(estr_both.refdate, d) for d in dates_new]
print(values)

plt.plot(dates_final, df_final, marker="^", label="bootstrapped")
plt.plot(dates_new, values, label="interpolated")

plt.xlabel("year")
plt.ylabel("DF")
plt.legend()

# continuous compounding
plt.figure(4)
dcc = DayCounter(estr_both.daycounter)
zero_final = []
for i in range(1, len(df_final)):
    delta_t = dcc.yf(estr_both.refdate, dates_final[i])  # float((dates_new[i] - estr.refdate).days) / 365.0
    zero_final.append(-math.log(df_final[i]) / delta_t)
zero_final.insert(0, zero_final[0])  # adding first entry to account for division by zero for dt

zero_interp = []
for i in range(1, len(values)):
    delta_t = dcc.yf(estr_both.refdate, dates_new[i])  # float((dates_new[i] - estr.refdate).days) / 365.0
    zero_interp.append(-math.log(values[i]) / delta_t)
zero_interp.insert(0, zero_interp[0])

plt.plot(dates_final, zero_final, marker="^", label="bootstrapped")
# plt.scatter(end_date_deposits, quotes_deposits)
plt.plot(dates_new, zero_interp, label="interpolated")

plt.xlabel("year")
plt.ylabel("zero rates")
plt.legend()
plt.show()

In [ ]:
model_quotes = []
pricing_params = {"fixing_grace_period": 0.0, "set_rate": True, "desired_rate": 1.0}
curves_dict = {"discount_curve": estr_both, "discount_curve":estr_both}




for i in range(len(instruments_both)):

    model_quote = get_quote(ref_date, instruments_both[i], curves_dict )

    print(model_quote)
    model_quotes.append(val)

In [ ]:
for i in range(len(quotes_both)):

    per_diff = (model_quotes[i] - quotes_both[i])/quotes_both[i] * 100
    print(f"i:{i} date:{dates_final[i+1]}  percent diff: {per_diff}")